# SafeStride road-surface training

This notebook rebuilds the ROS-compatible nine-class road-surface model. It uses grouped train/validation/test splits, fine-tunes MobileNetV3-Small for Raspberry Pi deployment, evaluates every class, and approves a production artifact only when the held-out test gate passes.

The executable training tools are embedded for standalone use and are also displayed below. Dataset preparation and epoch checkpoints persist in Google Drive, so a compatible interrupted run resumes instead of restarting. Select a GPU runtime before running all cells.


In [ ]:
!nvidia-smi
!pip -q install -U scikit-learn pandas pillow requests tqdm huggingface_hub

import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then reconnect.'
try:
    from torch.ao.quantization.quantize_fx import convert_fx, prepare_fx
except ImportError as error:
    raise RuntimeError('This export pipeline requires PyTorch FX static quantization support.') from error
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import base64
import gzip
import subprocess

# The standalone notebook writes the exact checked-in tools to Colab.
REPO_DIR = Path('/content/SafeStrideTraining')
TOOLS_DIR = REPO_DIR / 'tools'
TOOLS_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_TOOLS = {
    "road_surface_labels.py": "H4sIAAAAAAACCrVWTY/TMBC991dYPkAqthVwA2mRVoA48HHYrLggsJxk0pp14mA72y2w/52xHacpMftxoIfGdmaeZ968mZZSmm+5hopIXoAkpWqt5qUltdKk6wspSqIVr1am1zUvgeBb0Yp2Qypu+ZpSuljUWjWEsbq3vQbGiGg6pS3hbasst0K1ZrEYzjQsFouLs/N3by/Y6w9nef42J6ckWxD8UNMoZbes41dQ0ZNwplW/+euokKq8PD7aaNzJuGv6ilVC27jvW2/MGnF9cNmBPcZwB4NlPDKt2jFRAu6XGHcFNSkl8JZZuLbZFZc9vCSq+A6lXZLVK2Ksfhk8Kf2kdMOl+AmBWEM8S3YLkVXHnwFrAofOy6EiG4gSsJdrXIouW66l2oHOlt5KA9Lc4mNt+iLT9Ms3vvr5dPXi6xN6QijDL4cTffEght7wjmlTVswHlGm++zt48pt8Ui2MOXzkHTnPX795bJwuKuFquWq4BS24XBm4woXdxwStIjmvIcd7KzhkFXR1OmUOrw65WHUJrXE5g8284dp0UlgfdjARdbR6RH5RX4xQGP+UvdnSmxDwhJxD5UIMwjAsL95TKCWzCR6eOpydy8ktur6qpL+ilkpVKIWbZUQYxJFAifLz0nOPoD5CgWu7HSHqIY55uFPtDXbxNmQWpmqNUPFSItqBoDnsYJLgMR3nHGFspQRGqYpCgrGoGE8d3qWHBc4HXx6wHr7QorxMwU9bOXEDN92WSw+B+ivRC6Yox8ZhejjbDVbOZ3VdgpTQ2qnTkUqmA2cW3HT0TDvPtcikpbBvANs4jMcrYcIwG/bM7rtxRpwcvfnR43iw+8PL27ow95fkwfWzwGZDXJywFRlghiY8tF2c10eNN40qiDK6J82Gl6N8IyZK7tdQZubrbw51Z6MgfPXvWfQx2tNxXNO55/EgT0Z1t2Rixt78IJGJbp65r+d+xZ76NT7+p4S0vYxcP2w4u5+T84v3kYJVzA1bFj2V3pOWN/ecxUjNMIEtTgOzE3abRTqZp2aZmLOz9G+FSUHMOEoihMKn/FNymvtH6aQQErJK16ncl9JJ/ME1gmoDK6UFCg1H+gAT/wHMWveOKo0adxL2tvesS5zMox/B/3ihTW+BSvFLDz9W//S7g9U/4b/vNHkKAAA=",
    "train_road_surface.py": "..."
}
for filename, encoded in EMBEDDED_TOOLS.items():
    source = gzip.decompress(base64.b64decode(encoded))
    (TOOLS_DIR / filename).write_bytes(source)
print('Prepared standalone training tools in', TOOLS_DIR)


## Inspect the exact implementation

The short training cell later in the notebook is only a process launcher. These cells display the exact source that it executes: dataset and loaders, optimizer/training loop, and post-training static INT8 calibration/conversion.


In [ ]:
import importlib
import inspect
import sys
from IPython.display import Code, Markdown, display

sys.path.insert(0, str(TOOLS_DIR))
sys.modules.pop('train_road_surface', None)
trainer = importlib.import_module('train_road_surface')

for name in ('RoadSurfaceDataset', 'build_loaders', 'build_calibration_loader'):
    display(Markdown(f'### `{name}`'))
    display(Code(inspect.getsource(getattr(trainer, name)), language='python'))


In [ ]:
for name in ('build_model', 'run_epoch', 'train_candidate', 'choose_candidate'):
    display(Markdown(f'### `{name}`'))
    display(Code(inspect.getsource(getattr(trainer, name)), language='python'))


In [ ]:
for name in ('quantize_model', 'passes_export_gate'):
    display(Markdown(f'### `{name}`'))
    display(Code(inspect.getsource(getattr(trainer, name)), language='python'))


## What optimization is actually applied

- Training is FP32 transfer learning with AMP on the GPU.
- Quantization is FX post-training static quantization for the QNNPACK ARM backend. Weights use qint8 and activations use quint8.
- Calibration uses a deterministic, class-balanced subset of the train split with resize/normalize only. Validation and test images are never calibration inputs.
- INT8 is deployed only if validation macro-F1 drops by at most 0.015 and worst per-class recall drops by at most 0.05.
- Pruning is not applied. Unstructured zeroing does not automatically accelerate dense Raspberry Pi kernels, so the pipeline records `pruning: none` rather than claiming an unsupported speedup.


In [ ]:
import os
from google.colab import drive

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('Hugging Face authentication enabled.')
else:
    print('Using public Hugging Face access. Optional: add HF_TOKEN in Colab Secrets for higher rate limits.')

drive.mount('/content/drive')
WORK_DIR = Path('/content/drive/MyDrive/SafeStride/road_surface_training_v2')
RUN_DIR = Path('/content/drive/MyDrive/SafeStride/road_surface_mobilenet_v3_small_v1')
EXPORT_DIR = RUN_DIR / 'export'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
WORK_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the completed manifest from the earlier three-model notebook once.
PREPARED_MANIFEST = WORK_DIR / 'prepared' / 'dataset_manifest.csv'
LEGACY_EXPORT_ROOT = Path('/content/drive/MyDrive/SafeStride/road_surface_exports_v2')
legacy_manifests = (
    sorted(LEGACY_EXPORT_ROOT.glob('*/dataset_manifest.csv'), key=lambda path: path.stat().st_mtime)
    if LEGACY_EXPORT_ROOT.is_dir()
    else []
)
REUSE_MANIFEST = None if PREPARED_MANIFEST.is_file() else (legacy_manifests[-1] if legacy_manifests else None)
print('cache:', WORK_DIR)
print('exports:', EXPORT_DIR)
print('checkpoints:', CHECKPOINT_DIR)
if REUSE_MANIFEST:
    print('reusing earlier prepared manifest:', REUSE_MANIFEST)


## Train, validate, calibrate, and export

The default run keeps all nine ROS classes. Training requires at least 60 valid images per class and warns below the recommended 250. Validation and test each require 10 independent examples per class. The final held-out test gate requires macro F1 >= 0.75 and recall >= 0.55 for every class.

MobileNetV3-Small is the only trained backbone in this practical Raspberry Pi run. The maximum is 15 fine-tuning epochs, validation macro-F1 controls LR decay and four unimproved fine-tuning epochs trigger early stopping. A checkpoint is atomically written to Drive after every epoch.

RSCD itself has no `block_paved` or `unpaved_mixed` label, so those two RSCD warnings are expected; the other public sources supply those classes. The prepared dataset manifest is cached and reused after its first successful preparation.


In [ ]:
print('Starting dataset preparation and training.', flush=True)
command = [
    sys.executable, '-u', str(TOOLS_DIR / 'train_road_surface.py'),
    '--work-dir', str(WORK_DIR),
    '--export-dir', str(EXPORT_DIR),
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--resume',
    '--models', 'mobilenet_v3_small',
    '--batch-size', '128',
    '--finetune-epochs', '15',
    '--early-stop-patience', '4',
    '--rscd-download-workers', '4',
    '--quantize-int8',
]
if REUSE_MANIFEST is not None:
    command.extend(['--dataset-manifest', str(REUSE_MANIFEST)])
print('command:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)


In [ ]:
import json

manifest = json.loads((EXPORT_DIR / 'model_manifest.json').read_text(encoding='utf-8'))
print(json.dumps({
    'model_name': manifest['model_name'],
    'deployment_approved': manifest['deployment_approved'],
    'deployment_gate_reasons': manifest['deployment_gate_reasons'],
    'quantization': manifest['quantization'],
    'quantization_report': manifest['quantization_report'],
    'pruning': manifest['pruning'],
    'size_mb': round(manifest['artifact']['size_bytes'] / 1024 / 1024, 2),
    'validation_macro_f1': manifest['metrics']['validation']['macro_f1'],
    'test_macro_f1': manifest['metrics']['test']['macro_f1'],
    'test_recall': manifest['metrics']['test']['per_class_recall'],
}, indent=2))


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/safestride_road_surface_model', 'zip', EXPORT_DIR)
files.download(archive)
